# NLSV1 Phase 5 — LoRA Fine-Tune on Kaggle T4
Trains two LoRA adapters on Gemma-2-2B:
- **Adapter A**: English CoT baseline
- **Adapter B**: Neuralese reasoning chain

Run all cells top to bottom. Saves adapters to `/kaggle/working/`.

In [ ]:
# ── Cell 1: Install deps ─────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.44.2', 'peft==0.12.0', 'accelerate==0.33.0',
    'datasets', 'huggingface_hub'], check=True)
print('Deps installed.')

In [ ]:
# ── Cell 2: Sanity check — GPU must be visible ────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected — enable T4 in Notebook Settings before continuing.')
print('Device:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Cell 3: Download corpus from HuggingFace ─────────────────────────────
import os
from huggingface_hub import login
from datasets import load_dataset
import json
from pathlib import Path

import os
HF_TOKEN = os.environ['HF_TOKEN']
login(token=HF_TOKEN, add_to_git_credential=False)

ds = load_dataset('ALBYJ07/neuralese-v1-corpus', split='train', token=HF_TOKEN)
print(f'Corpus loaded: {len(ds)} examples')
print('Columns:', ds.column_names)
print('Sample:', ds[0])

In [ ]:
# ── Cell 4: Build stratified 90/10 train/test split ──────────────────────
import random
random.seed(42)

def difficulty(ex):
    cot = ex.get('original_cot', '')
    w = len(cot.split())
    if w < 60:   return 'easy'
    if w < 120:  return 'medium'
    return 'hard'

# Group by difficulty
buckets = {'easy': [], 'medium': [], 'hard': []}
for ex in ds:
    buckets[difficulty(ex)].append(ex)

train_data, test_data = [], []
for bucket, examples in buckets.items():
    random.shuffle(examples)
    n_test = max(1, int(len(examples) * 0.10))
    test_data.extend(examples[:n_test])
    train_data.extend(examples[n_test:])

random.shuffle(train_data)
random.shuffle(test_data)

print(f'Train: {len(train_data)}  Test: {len(test_data)}')
for b in ['easy','medium','hard']:
    tr = sum(1 for e in train_data if difficulty(e)==b)
    te = sum(1 for e in test_data  if difficulty(e)==b)
    print(f'  {b:6s}: train={tr}  test={te}')

In [ ]:
import os
HF_TOKEN = os.environ['HF_TOKEN']
# ── Cell 5: Tiny gate run — 50 examples, 1 epoch, Neuralese ──────────────
# MUST pass before running full training.
# Confirms: no crash, no CPU fallback, no NaN loss.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

MODEL_ID   = 'google/gemma-2-2b'
HF_TOKEN   = HF_TOKEN
MAX_SEQ    = 512
WORK_DIR   = Path('/kaggle/working')

NEURALESE_TOKENS = [
    'ARITH_EQUALS', 'NUMERIC_RESULT', 'ARITH_MULTIPLY', 'ARITH_ADD',
    'ARITH_DIVIDE', 'ARITH_SUBTRACT', 'CAUSAL_REQUIRES', 'TEMPORAL_CHANGE',
    'CAUSAL_ENABLES', 'TEMPORAL_DURING', 'TEMPORAL_AFTER', 'TEMPORAL_BEFORE',
    'CAUSAL_CONTRIBUTES', 'RUNNING_TOTAL', 'LOGICAL_AND', 'IS_EQUIVALENT_TO',
    'THEREFORE', 'LOGICAL_NOT', 'LOGICAL_IMPLIES', 'FORALL', 'EXISTS',
]

def load_model_and_tokenizer(lora_rank=8):
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    num_added = tok.add_tokens(NEURALESE_TOKENS, special_tokens=False)
    print(f'Added {num_added} Neuralese tokens')

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16,
        device_map='auto', token=HF_TOKEN,
    )
    model.gradient_checkpointing_enable()

    if num_added > 0:
        orig_tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
        subword_ids = [orig_tok.encode(s, add_special_tokens=False) for s in NEURALESE_TOKENS]
        model.resize_token_embeddings(len(tok))
        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            for sym, new_id, sw_ids in zip(
                NEURALESE_TOKENS, tok.convert_tokens_to_ids(NEURALESE_TOKENS), subword_ids
            ):
                emb[new_id] = emb[sw_ids].mean(dim=0)

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=lora_rank, lora_alpha=lora_rank*2,
        target_modules=['q_proj','k_proj','v_proj','o_proj'],
        lora_dropout=0.05, bias='none',
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return tok, model

def make_dataset(examples, tokenizer, adapter='neuralese'):
    if adapter == 'neuralese':
        texts = [
            f"<problem>{e['question']}</problem>\n"
            f"<reasoning>\n{e['neuralese_chain']}\n</reasoning>\n"
            f"<answer>{e['answer']}</answer>"
            for e in examples
        ]
    else:  # english CoT
        texts = [
            f"<problem>{e['question']}</problem>\n"
            f"<reasoning>\n{e['original_cot']}\n</reasoning>\n"
            f"<answer>{e['answer']}</answer>"
            for e in examples
        ]
    ds = Dataset.from_dict({'text': texts})
    return ds.map(
        lambda x: tokenizer(x['text'], truncation=True, max_length=MAX_SEQ, padding=False),
        batched=True, remove_columns=['text']
    )

def run_training(tokenizer, model, train_ds, output_dir, epochs=3, batch=1, grad_acc=8, label=''):
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=grad_acc,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        eval_strategy='no',
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=0,
        gradient_checkpointing=True,
        ddp_find_unused_parameters=False,
        label_names=[],
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    print(f'\n=== Training {label} ===')
    result = trainer.train()
    print(f'Final loss: {result.training_loss:.4f}')
    return trainer, result

# ── Tiny gate run ─────────────────────────────────────────────────────────
print('=== TINY GATE RUN: 50 examples, 1 epoch ===')
gate_examples = train_data[:50]
tok, model = load_model_and_tokenizer(lora_rank=8)
gate_ds = make_dataset(gate_examples, tok, adapter='neuralese')
_, gate_result = run_training(tok, model, gate_ds,
    output_dir=WORK_DIR/'gate_run', epochs=1, batch=1, grad_acc=4, label='GATE')

import math
if math.isnan(gate_result.training_loss):
    raise RuntimeError('NaN loss — do not proceed to full run.')
print(f'\nGATE PASSED. Loss={gate_result.training_loss:.4f}. Proceeding to full run.')
del model; torch.cuda.empty_cache()

In [ ]:
# ── Cell 6: Full run — Adapter B (Neuralese) ─────────────────────────────
print('=== ADAPTER B: Neuralese reasoning ===')
tok_b, model_b = load_model_and_tokenizer(lora_rank=8)
train_ds_b = make_dataset(train_data, tok_b, adapter='neuralese')
trainer_b, result_b = run_training(
    tok_b, model_b, train_ds_b,
    output_dir=WORK_DIR/'adapter_B_neuralese',
    epochs=3, batch=1, grad_acc=8, label='Adapter B (Neuralese)'
)
model_b.save_pretrained(str(WORK_DIR/'adapter_B_neuralese'/'final'))
tok_b.save_pretrained(str(WORK_DIR/'adapter_B_neuralese'/'final'))
print('Adapter B saved.')
del model_b; torch.cuda.empty_cache()

In [ ]:
# ── Cell 7: Full run — Adapter A (English CoT baseline) ──────────────────
print('=== ADAPTER A: English CoT baseline ===')
tok_a, model_a = load_model_and_tokenizer(lora_rank=8)
train_ds_a = make_dataset(train_data, tok_a, adapter='english')
trainer_a, result_a = run_training(
    tok_a, model_a, train_ds_a,
    output_dir=WORK_DIR/'adapter_A_english',
    epochs=3, batch=1, grad_acc=8, label='Adapter A (English CoT)'
)
model_a.save_pretrained(str(WORK_DIR/'adapter_A_english'/'final'))
tok_a.save_pretrained(str(WORK_DIR/'adapter_A_english'/'final'))
print('Adapter A saved.')
del model_a; torch.cuda.empty_cache()

In [ ]:
# ── Cell 8: Evaluation ───────────────────────────────────────────────────
import re
from peft import PeftModel

def extract_answer(text):
    m = re.search(r'<answer>\s*([\d,\.\-]+)', text)
    if m:
        return m.group(1).replace(',', '').strip()
    nums = re.findall(r'[\d,\.]+', text)
    return nums[-1].replace(',','') if nums else ''

def evaluate_adapter(adapter_path, test_examples, adapter_type, tokenizer):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN
    )
    base.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    tokenizer.padding_side = 'left'

    correct = 0
    total_reasoning_tokens = 0
    results_by_difficulty = {'easy': [0,0], 'medium': [0,0], 'hard': [0,0]}

    for ex in test_examples:
        prompt = f"<problem>{ex['question']}</problem>\n<reasoning>\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        total_reasoning_tokens += len(out[0]) - inputs['input_ids'].shape[1]

        pred = extract_answer(generated)
        gold = str(ex['answer']).replace(',', '').strip()
        is_correct = pred == gold
        if is_correct:
            correct += 1
        d = difficulty(ex)
        results_by_difficulty[d][0] += int(is_correct)
        results_by_difficulty[d][1] += 1

    n = len(test_examples)
    print(f'\n=== {adapter_type} Evaluation ===')
    print(f'  Accuracy:              {correct}/{n}  ({100*correct/n:.1f}%)')
    print(f'  Mean reasoning tokens: {total_reasoning_tokens/n:.1f}')
    for d, (c, t) in results_by_difficulty.items():
        if t > 0:
            print(f'  Accuracy [{d:6s}]:    {c}/{t}  ({100*c/max(t,1):.1f}%)')
    del model; torch.cuda.empty_cache()
    return correct/n, total_reasoning_tokens/n

# Load tokenizer B (has Neuralese vocab) for adapter B eval
tok_eval_b = AutoTokenizer.from_pretrained(
    str(WORK_DIR/'adapter_B_neuralese'/'final'), token=HF_TOKEN
)
tok_eval_b.pad_token = tok_eval_b.eos_token

# Load tokenizer A (also has Neuralese vocab — same base) for adapter A eval
tok_eval_a = AutoTokenizer.from_pretrained(
    str(WORK_DIR/'adapter_A_english'/'final'), token=HF_TOKEN
)
tok_eval_a.pad_token = tok_eval_a.eos_token

acc_b, tok_b_mean = evaluate_adapter(
    WORK_DIR/'adapter_B_neuralese'/'final', test_data, 'Adapter B (Neuralese)', tok_eval_b
)
acc_a, tok_a_mean = evaluate_adapter(
    WORK_DIR/'adapter_A_english'/'final',    test_data, 'Adapter A (English CoT)', tok_eval_a
)

print('\n' + '='*50)
print('FINAL COMPARISON')
print('='*50)
print(f'  {"Adapter":30s} {"Accuracy":>10s} {"Mean tokens":>12s}')
print(f'  {"A — English CoT":30s} {100*acc_a:>9.1f}% {tok_a_mean:>12.1f}')
print(f'  {"B — Neuralese":30s} {100*acc_b:>9.1f}% {tok_b_mean:>12.1f}')
print(f'  Token ratio B/A: {tok_b_mean/tok_a_mean:.3f}')